# CoLA Advection

In this notebook related to artificial data generation, we are going to advect the actomyosin cortex based on a velocity field. 

1. [Generating Actomyosin Cortices](./artificial_cells.ipynb) was used to generate synthetic microscopy data of actomyosin cortices.
2. [Generating CoLA Cuts](./artificial_cola.ipynb) was used to introduce a quasi-random cut to simulate where CoLA could happen in real data.
3. [Generating Velocity Fields](./cola_velocity_field.ipynb) was used to generate a synthetic velocity field basedon the cut.

We are now going to integrate these three. While conceptually easy, the concept of **advection** lies at the heart of this notebook.

In [ ]:
import matplotlib as mpl
import numpy as np

from IPython.display import HTML
from tqdm import tqdm

# Increase the embed size limit
mpl.rcParams['animation.embed_limit'] = 400_000_000  # 400 MB

Make sure to run `pip install -e .` before running the cell below. This command will make sure `diet4cola` is installed as a module in your current Python/Conda environment.

In [ ]:
from diet4cola.cut import CoLACut
from diet4cola.mask import mask_on_condition
from diet4cola.operations import invert
from diet4cola.velocity import prior_exponential, compute_velocity_field_multi
from diet4cola.sdf import sdf_box, compute_rounded_sdf_multi

from diet4cola.utils import *

## Synthetic Actomyosin Cortex

**! Disclaimer &mdash; Uncommend animations if you wish to view them, they take quite some time to load for longer simulations (i.e. smaller timesteps)**

As we have done many times before, we create (or in this particular case load) a synthetic actomyosin cortex and introduce an synthetic CoLA cut.

In [ ]:
# Cell parameters
seed = 42
width = 512
height = 512

np.random.seed(seed)
off_x = 256 + (np.random.randint(32) - 16)
off_y = 256 + (np.random.randint(32) - 16)

In [ ]:
actomyosin_layer = load_cortex('./data/cortex_1.npy')

In [ ]:
plot_2d_array(actomyosin_layer, 'Synthetic Actomyosin Cortex')

In [ ]:
cola_cut = CoLACut((off_x, off_y), 32, 150, seed)

In [ ]:
plot_synthetic_cut(actomyosin_layer, 
                   cola_cut.cut_center, 
                   cola_cut.cut_origin,
                   cola_cut.cut_destination)

## Velocity Field

In the [previous notebook](./cola_velocity_field.ipynb) we created a velocity field. From physics, we know that velocity determines how a point moves in space over time. Imagine the velocity $\bold{v}(\bold{r}, t)$ of $\bold{r} = (x, y)$ can be split into the velocities for each component of $\bold{r}$.

* $v_x(\bold{r}, t)$ gives the displacement in the direction of the $x$-axis.
* $v_y(\bold{r}, t)$ gives the displacement in the direction of the $y$-axis.

Let's start by recreating the velocity field from earlier. We will tweak the timescale after we are done, since the current velocity field gives a pretty accurate intuition.

In [ ]:
t_min = 0
t_max = 20

# From t_min, t_max, iterations, compute step
time_step = 0.5

# Create time points
timepoints = np.arange(t_min, t_max + time_step, time_step)
iterations = len(timepoints)

normalized_timepoints = timepoints / t_max

Let's set the actual parameters for the SDF and velocity field. After this, we simply generate the SDF and velocity field over time.

In [ ]:
k = 0.1
parameter = 40
radius = 10
alpha = 10

initial_velocities = np.array([alpha * np.exp(-t * k) for t in timepoints])
inverted_initial_velocities_norm = invert(initial_velocities / alpha)
parameters = np.array([parameter] * iterations)

radii = [0] * iterations
widths = [0] * iterations

# Create proper radii/widths
for i in range(iterations):
    if i == 0:
        radii[i] = 0
        widths[i] = 0
        continue

    #radii[i] = radius * inverted_initial_velocities_norm[i]
    widths[i] = widths[i - 1] + time_step * initial_velocities[i]

plot_curve(initial_velocities, timepoints, 'Initial velocities')
plot_curve(inverted_initial_velocities_norm, timepoints, 'Initial velocities (Normalized)')
plot_curve(parameters, timepoints, 'Parameters')
plot_curve(radii, timepoints, 'Radii')
plot_curve(widths, timepoints, 'Widths')

In [ ]:
sdf_fields = compute_rounded_sdf_multi(512, 512, cola_cut.cut_origin, cola_cut.cut_destination, widths, radii, False, sdf_box, iterations)

In [ ]:
sdf_animation = animate_2d_data(sdf_fields, None, None, 100, 'RdBu_r', 'SDF (over Time)')

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(sdf_animation.to_jshtml())
plt.close(sdf_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
masked_sdf_fields = np.array([mask_on_condition(sdf < 0, 0, 1) for sdf in sdf_fields])
clipped_rounded_sdf_fields = np.array([mask_on_condition(sdf < 0, 0, sdf) for sdf in sdf_fields])

In [ ]:
masked_sdf_anim = animate_2d_data(masked_sdf_fields, None, None, 100, 'gray', 'SDF Mask (over Time)')
clipped_sdf_anim = animate_2d_data(clipped_rounded_sdf_fields, None, None, 100, 'RdBu_r', 'Clipped SDF (over Time)')

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(masked_sdf_anim.to_jshtml())
plt.close(masked_sdf_anim._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(clipped_sdf_anim.to_jshtml())
plt.close(clipped_sdf_anim._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
velocity_fields = compute_velocity_field_multi(clipped_rounded_sdf_fields, prior_exponential, timepoints, initial_velocities, parameters, iterations)
masked_velocity_fields = np.array([vf * msdf for (vf, msdf) in zip(velocity_fields, masked_sdf_fields)])

## Backward Advection

Before diving into the code necessary for backward advection, let's go over what backward advection actually is. In computational fluid dynamics, **advection** refers to the transport of a quantity (in this case, the 'density' of the actomyosin cortex) by the flow of a velocity field. 

If we denote

* $\phi(\bold{r}, t)$ as a scalar quantity (e.g. density)
* $\bold{v}(\bold{r}, t)$ as the velocity field (as explained earlier)

then advection is governed by 

$$ \frac{\partial \phi}{\partial t} + \bold{v} \cdot \nabla \phi = 0 $$

This means that the value of $\phi$ attached to each "fluid" parcel remains constant as it moves &mdash; it's simply *carried* by the flow without being created or destroyed. The quantity $\phi$ is **conserved along the flow**.

In **forward advection**, we compute how the quantity moves *with* the velocity. However, this causes significant numerical instabilities since we are trying to predict the future values from the current ones.

**Backward advection** (i.e. a semi-Lagrangian method) instead traces backward along the velocity field to find *where a fluid parcel came from*. This method is stable, even for large time steps and is thus ideal for our use case.

### Idea behind Backward Advection

To compute $\phi^{n+1}(\bold{r})$, instead of moving the quantity forward, we ask ourselves:

> "Where was the fluid particle currently at $\bold{r}$ located at the previous time step?"

Mathematically speaking;

$$ \bold{r}_{\text{prev}} = \bold{r} - \Delta t \bold{v}(\bold{r})$$

Then we sample $\phi^n(\bold{r}_{\text{prev}})$, &mdash; usually via interpolation since $\bold{r}_{\text{prev}}$ isn't perfectly located on a grid point &mdash; and assign that value to the current cell:

$$ \phi^{n+1}(\bold{r}) = \phi^n(\bold{r}_{\text{prev}})$$

This is the core of the **semi-Lagrangian advection schema**, which we will now be implementing on our 2D grid.

### 2D Numerical Derivation

Let's unpack how this works numerically in our situation. We are given

* A 2D scalar field, the *actomyosin cortex* which we will denote $\phi_{\text{amc}}$.
* A 2D velocity field which was synthetically generated which we will call $\bold{v}$ where $\bold{v}(\bold{r}) = (v_x(\bold{r}), v_y(\bold{r}))$
* A timestep given by the variable `step`, which in the following derivation will be denoted $dt$ for brevity.

For each grid cell $(i, j)$:

1. Compute the **backtraced position**:

$$ 
\begin{cases}
    x_{\text{prev}} = i - dt \cdot v_x(i, j)\\
    y_{\text{prev}} = j - dt \cdot v_y(i, j)
\end{cases}
$$

2. Use **bilinear interpolation** to find $\phi(x_{\text{prev}}, y_{\text{prev}})$:
    * Find the integer grid coordinates surrounding $(x_{\text{prev}}, y_{\text{prev}})$. This simply is the floor/ceil of $x_{\text{prev}}$, $y_{\text{prev}}$.
    * Interpolate between those four neighbouring grid values.

3. Store the interpolated value in the new field

4. **!** Handle boundary conditions properly &mdash; since in our case, in theory, the actomyosin cortex can move "outside" of the image frame, we can simply discard the update rule (i.e. mass **CAN** leave our bounds!)

**Bilinear Interpolation** &mdash; We first create a method to bilinearly interpolate fractional $x$ and $y$ values.

In [ ]:
def interp_bilinear(data: np.ndarray,
                    x: float, 
                    y: float) -> float:
    # Compute possible grid coordinates
    x_0 = np.floor(x).astype(int)
    x_1 = x_0 + 1
    y_0 = np.floor(y).astype(int)
    y_1 = y_0 + 1

    # Clamp values to the grid (to prevent out of bounds!)
    height, width = data.shape  
    x_0 = np.clip(x_0, 0, width - 1)
    x_1 = np.clip(x_1, 0, width - 1)
    y_0 = np.clip(y_0, 0, height - 1)
    y_1 = np.clip(y_1, 0, height - 1)

    # Interpolation weights
    w_x = x - x_0
    w_y = y - y_0

    # Linearly interpolate first along x then along y
    top = (1 - w_x) * data[y_0, x_0] + w_x * data[y_0, x_1]
    bottom = (1 - w_x) * data[y_1, x_0] + w_x * data[y_1, x_1]
    return (1 - w_y) * top + w_y * bottom


**Backward Advection** &mdash; Next we create a method that performs the semi-Lagrangian backward advection step. 

In [ ]:
def advect_backward(phi: np.ndarray,
                    v_x: np.ndarray,
                    v_y: np.ndarray,
                    magnitude: np.ndarray,
                    dt: float) -> float:
    n_y, n_x = phi.shape
    x, y = np.meshgrid(np.arange(n_x), np.arange(n_y))

    # 1 - Backtrace positions
    x_prev = x - (dt * v_x * magnitude)
    y_prev = y - (dt * v_y * magnitude)

    # 2 - Interpolate old phi bilinearly at the backtraced position
    phi_new = interp_bilinear(phi, x_prev, y_prev)

    return phi_new

### Noise Field Example

Let's see this in action using a simple example! We will create a small simplex noise field which we will move in a swirling motion.

In [ ]:
from diet4cola.noise import *

In [ ]:
ex_noise_field = fractal_snoise(get_snoise_generator(seed), width, height, 0.01, 2)

plot_2d_array(ex_noise_field, 'Example Noise Field', 'RdBu_r')

As the velocity field, let's choose the noise field itself! This means that values 

In [ ]:
def directional_gradient(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    # f is a 2D numpy array
    fx = np.zeros_like(data)
    fy = np.zeros_like(data)

    fx[1:-1, :] = (data[2:, :] - data[:-2, :]) / 2.0
    fy[:, 1:-1] = (data[:, 2:] - data[:, :-2]) / 2.0

    magnitude = np.sqrt(fx ** 2 + fy ** 2)
    direction = np.arctan2(fy, fx)  # angle in radians
    return fx, fy, magnitude, direction

ex_noise_velocity = ex_noise_field.copy()
ex_noise_vx, ex_noise_vy, ex_magnitude, ex_noise_dir = directional_gradient(ex_noise_velocity)

ex_min = np.minimum(np.min(ex_noise_vx), np.min(ex_noise_vy))
ex_max = np.maximum(np.max(ex_noise_vx), np.max(ex_noise_vy))

plot_2d_array_comparison(ex_noise_vx, ex_noise_vy, 'Velocity (x-comp)', 'Velocity (y-comp)', cmap='RdBu_r', min=ex_min, max=ex_max)
plot_2d_array(ex_noise_dir, 'Noise Direction', min=np.min(ex_noise_dir), max=np.max(ex_noise_dir))

Note the large time step here! This is simply to show an actual change (and backward advection can luckily handle such large steps quite well numerically speaking.)

In [ ]:
ex_noise_fields = []
ex_noise_fields.append(ex_noise_field)

for i in tqdm(range(25)):
    ex_noise_field_i = advect_backward(ex_noise_fields[i], ex_noise_vx, ex_noise_vy, 1, 50)
    ex_noise_fields.append(ex_noise_field_i)

ex_noise_fields = np.array(ex_noise_fields)

In [ ]:
ex_animation = animate_2d_data(ex_noise_fields, None, None, 100, 'RdBu_r', 'Backward Advection Example')

In [ ]:
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(ex_animation.to_jshtml())
plt.close(ex_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim

## Backward Advection on Synthetic CoLA Cut

This is really where the magic happens! We will use the velocity field to perform backward advection on the actomyosin cortex. Let's visualize the velocity field over time once more.

In [ ]:
vf_animation = animate_2d_data(velocity_fields, None, None, 100, 'viridis', 'Velocity (unmasked) over Time')
vfm_animation = animate_2d_data(masked_velocity_fields, None, None, 100, 'viridis', 'Velocity (masked) over Time')

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(vf_animation.to_jshtml())
plt.close(vf_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(vfm_animation.to_jshtml())
plt.close(vfm_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

Just as in the example, we will now compute the gradient of the velocity fields. This will give us the directionality we need!

In [ ]:
def directional_gradients_multi(datas: np.ndarray) -> np.ndarray:
    gradients = np.array([directional_gradient(data) for data in datas])
    return gradients[:, :2, :, :]

Let's quickly visualize what all these gradients are doing over time, so that we get a better idea of what we are actually trying to do.

### Velocity Gradients

In [ ]:
velocity_gradients = directional_gradients_multi(velocity_fields)
velocity_dxs = velocity_gradients[:, 0, :, :] * -1 # Reverse gradient to go outward
velocity_dys = velocity_gradients[:, 1, :, :] * -1 # Reverse gradient to go outward

velocity_grad_magnitudes = np.sqrt(velocity_dxs ** 2 + velocity_dys ** 2)

In [ ]:
'''
vel_dx_animation = animate_2d_data(velocity_dxs, None, None, 100, 'RdBu_r', 'Velocity dX (over Time)')
vel_dy_animation = animate_2d_data(velocity_dys, None, None, 100, 'RdBu_r', 'Velocity dY (over Time)')
vel_grad_animation = animate_2d_data(velocity_grad_magnitudes, None, None, 100, 'RdBu_r', 'Velocity Gradient Magnitude (over Time)')
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(vel_dx_animation.to_jshtml())
plt.close(vel_dx_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(vel_dy_animation.to_jshtml())
plt.close(vel_dy_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(vel_grad_animation.to_jshtml())
plt.close(vel_grad_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

### SDF Gradients

In [ ]:
sdf_gradients = directional_gradients_multi(clipped_rounded_sdf_fields)

sdf_dys = sdf_gradients[:, 0, :, :]
sdf_dxs = sdf_gradients[:, 1, :, :]

sdf_gradient_magnitudes = np.sqrt(sdf_dxs ** 2 + sdf_dys ** 2)

In [ ]:
'''
sdf_dx_animation = animate_2d_data(sdf_dxs, None, None, 100, 'RdBu_r', 'SDF dX (over Time)')
sdf_dy_animation = animate_2d_data(sdf_dys, None, None, 100, 'RdBu_r', 'SDF dY (over Time)')
sdf_grad_animation = animate_2d_data(sdf_gradient_magnitudes, None, None, 100, 'RdBu_r', 'SDF Gradient Magnitude (over Time)')
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(sdf_dx_animation.to_jshtml())
plt.close(sdf_dx_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(sdf_dy_animation.to_jshtml())
plt.close(sdf_dy_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

In [ ]:
'''
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(sdf_grad_animation.to_jshtml())
plt.close(sdf_grad_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim
'''

Let's quickly visualize some gradient maps of the SDF over time, since we will be using it to advect the actomyosin cortex over time.

In [ ]:
'''
animate_gradient_field(sdf_dxs, sdf_dys, 16, False, 100, 'viridis', 'SDF Gradient')
'''

This is exactly the gradient field we need in order to move our points &mdash; over time, only points inside the SDF are not affected. Keep in mind that this gradient field only shows the **direction** at each time point. For the magnitude of movement, we use the velocity fields we have precomputed earlier. We can now put this all together in creating the advected actomyosin cortex.

### Actomyosin Cortex Advection

Now, because of how our advection works, we will first need to multiply the cortices with the masks generated earlier to get a clean empty space in the space of the cut. We will slightly blur the sdf masks however, since otherwise we get a quite unnatural result.

In [ ]:
blurred_masked_sdf_fields = np.array([blur(sdf.astype(np.float64), (5, 5), 5) for sdf in masked_sdf_fields])

mask_animation = animate_2d_data(blurred_masked_sdf_fields, None, None, 100, 'gray', 'Blurred SDF Mask')

In [ ]:
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(mask_animation.to_jshtml())
plt.close(mask_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim

We now multiply the gradients in x and y directions by the magnitude.

In [ ]:
advected_cortices = []
advected_cortices.append(actomyosin_layer)

for i in tqdm(range(iterations - 1)):
    # First, advect in the direction of the SDF with the magnitude of the velocity field
    advected_cortex_i = advect_backward(advected_cortices[i], sdf_dxs[i], sdf_dys[i], velocity_fields[i], time_step)

    # Append to the advected cortices
    advected_cortices.append(advected_cortex_i)

advected_cortices = np.array(advected_cortices)

# Multiply each advected cortex with the blurred SDF masks
advected_cortices = mul(advected_cortices, blurred_masked_sdf_fields)

Let's animate our final result!

In [ ]:
cortex_animation = animate_2d_data(advected_cortices, None, None, 100, 'gray', 'Synthetic Actomyosin Example')

In [ ]:
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

html_anim = HTML(cortex_animation.to_jshtml())
plt.close(cortex_animation._fig)  # <-- closes the figure so the static plot doesn't show
html_anim

In [ ]:
from matplotlib.animation import FFMpegWriter

# Optional: configure ffmpeg writer
writer = FFMpegWriter(fps=10, metadata=dict(artist='Matthias Kovacic'), bitrate=1800)

# Save the animation as an mp4
cortex_animation.save("./video/synthetic_cortex_animation.mp4", writer=writer)

# Close the figure if you don't want it displayed
plt.close(cortex_animation._fig)